# The Tangent Portfolio by Continuation
In L7a, we found the long-only tangent portfolio $\mathcal{T}$ by solving the risky and risk-free problem, [SIM-2](CHEME-5660-L7b-Reference-SIM-Portfolio-Problems-Fall-2026.ipynb), at one growth target and rescaling the risky weights to sum to one. This notebook shows why that shortcut works and gives a continuation method that locates the tangent point itself, the target at which the investor neither lends nor borrows.

> __Learning Objectives:__
>
> By the end of this algorithm notebook, you should be able to:
> - **Derive the common risky direction:** Show that every solution of [SIM-2](CHEME-5660-L7b-Reference-SIM-Portfolio-Problems-Fall-2026.ipynb) below the tangent point is the same risky fund scaled by the risky fraction, and explain why that fund has the largest Sharpe ratio of any long-only portfolio.
> - **Follow the continuation:** Step the growth target upward until lending turns into borrowing, then bisect the crossing to a risky fraction of one.
> - **Assess convergence and stopping:** Explain why the risky fraction crosses one exactly once, and distinguish meeting the tolerance from reaching the iteration limit.

This notebook builds on [SIM-2](CHEME-5660-L7b-Reference-SIM-Portfolio-Problems-Fall-2026.ipynb), and its main result is [SIM-3](CHEME-5660-L7b-Reference-SIM-Portfolio-Problems-Fall-2026.ipynb). The [utility allocator example](CHEME-5660-L7b-Example-Utility-Allocator-Fall-2026.ipynb) runs the continuation and checks it against the single solve.

___


## One risky direction
Let $\hat{\mathbf{g}}_{\mathrm{SIM}}$ be the SIM mean vector, in inverse years, $\hat{\mathbf{\Sigma}}_{g,\mathrm{SIM}}$ its positive-definite covariance matrix, and $g_{f}$ the risk-free rate. Write the excess-growth vector as $\mathbf{e}=\hat{\mathbf{g}}_{\mathrm{SIM}}-g_{f}\mathbf{1}$ and the excess target as $s=g_{\star}-g_{f}>0$. [SIM-2](CHEME-5660-L7b-Reference-SIM-Portfolio-Problems-Fall-2026.ipynb) chooses the wealth weights $\mathbf{w}$ of the risky holdings to solve:

$$
\underset{\mathbf{w}}{\operatorname{minimize}}\quad \mathbf{w}^{\top}\hat{\mathbf{\Sigma}}_{g,\mathrm{SIM}}\mathbf{w}
\qquad\text{subject to}\quad \mathbf{e}^{\top}\mathbf{w}\geq s,\qquad 0\leq w_{i}\leq 1.
$$

The risk-free fraction $w_{f}=1-\mathbf{1}^{\top}\mathbf{w}$ is whatever wealth is left, and it may be negative. Let $\mathbf{w}(s)$ denote the solution at excess target $s$, and let $\theta(s)=\mathbf{1}^{\top}\mathbf{w}(s)$ be its risky fraction.

> __Common risky direction:__
>
> __Setup.__ Assume at least one entry of $\mathbf{e}$ is positive, so every excess target up to $\max_{i}e_{i}$ is feasible, and consider targets at which no upper bound $w_{i}\leq1$ binds.
>
> __Result.__ The solutions lie on a ray through the origin:
>
> $$\boxed{\mathbf{w}(s)=\theta(s)\,\mathbf{w}_{\mathcal{T}},\qquad \theta(s)=\frac{s}{\mathbf{e}^{\top}\mathbf{w}_{\mathcal{T}}}=\frac{g_{\star}-g_{f}}{\hat{\mathbf{g}}_{\mathrm{SIM}}^{\top}\mathbf{w}_{\mathcal{T}}-g_{f}}}$$
>
> This is [SIM-3](CHEME-5660-L7b-Reference-SIM-Portfolio-Problems-Fall-2026.ipynb), written with the excess target $s$.
>
> where the tangent portfolio $\mathbf{w}_{\mathcal{T}}=\mathbf{w}(s)/\mathbf{1}^{\top}\mathbf{w}(s)$ is any solution rescaled to sum to one.
>
> __Derivation sketch.__ Drop the caps for the moment. Scaling $\mathbf{w}$ by $c>0$ scales the excess growth by $c$ and the variance by $c^{2}$, and the lower bounds $w_{i}\geq0$ still hold. So if $\mathbf{w}(s)$ is optimal at $s$, then $(s^{\prime}/s)\,\mathbf{w}(s)$ is feasible at $s^{\prime}$ with $(s^{\prime}/s)^{2}$ times the minimum variance at $s$. No feasible point at $s^{\prime}$ can have less variance, because scaling it by $s/s^{\prime}$ would beat the minimum at $s$. A positive-definite covariance matrix makes the minimizer unique, so $\mathbf{w}(s^{\prime})=(s^{\prime}/s)\,\mathbf{w}(s)$. The growth constraint holds with equality at the optimum, which gives $\theta(s)$. Restoring the caps changes nothing at targets where the uncapped solution satisfies them, which is the setup's assumption.

Two consequences follow:

* __One solve gives the fund:__ Any target below the tangent point, rescaled to sum to one, returns $\mathbf{w}_{\mathcal{T}}$. The next section states this shortcut as a procedure; the L7a and L7b examples use it.
* __The fund has the largest Sharpe ratio:__ Along the ray, the Sharpe ratio $\mathbf{e}^{\top}\mathbf{w}/\sqrt{\mathbf{w}^{\top}\hat{\mathbf{\Sigma}}_{g,\mathrm{SIM}}\mathbf{w}}$ is constant. Any other long-only portfolio with positive excess growth, scaled down to meet a small excess target, has at least the minimum variance there, so its Sharpe ratio is no larger. A portfolio with nonpositive excess growth has a nonpositive Sharpe ratio.

Along the ray, the risky fraction $\theta(s)$ grows linearly with the target. We lend when $\theta<1$, borrow when $\theta>1$, and hold $\mathcal{T}$ alone at $\theta=1$, which on the ray occurs at $g_{\star}=\hat{\mathbf{g}}_{\mathrm{SIM}}^{\top}\mathbf{w}_{\mathcal{T}}$, the expected growth rate of the tangent portfolio. A cap $w_{i}\leq1$ can bind on the ray only once $\theta\geq1/\max_{i}w_{\mathcal{T},i}$, which is at or beyond the tangent point, so the crossing comes before any cap binds unless the fund holds a single asset. A continuation method finds that crossing without knowing $\mathbf{w}_{\mathcal{T}}$ in advance.

___


## The one-solve shortcut
The result above says that, at any target below the tangent point, the solution of [SIM-2](CHEME-5660-L7b-Reference-SIM-Portfolio-Problems-Fall-2026.ipynb) is the same mix of firms held in a larger or smaller amount. Raising the target buys more of the same mix and lends less. The mix is the tangent portfolio, and the amount does not matter, so one solve is enough:

1. __Solve [SIM-2](CHEME-5660-L7b-Reference-SIM-Portfolio-Problems-Fall-2026.ipynb) once__ at a growth target $g_{\star}$ between $g_{f}$ and the tangent point. The examples use a target a quarter of the way from $g_{f}$ to the largest entry of $\hat{\mathbf{g}}_{\mathrm{SIM}}$.
2. __Add up the risky weights.__ The sum $\theta=\mathbf{1}^{\top}\mathbf{w}$ is the risky fraction, the amount held in the mix. At a target below the tangent point it is less than one, and the rest of the wealth is lent at $g_{f}$.
3. __Divide each weight by the sum.__ The rescaled vector $\mathbf{w}/\theta$ sums to one, and by [SIM-3](CHEME-5660-L7b-Reference-SIM-Portfolio-Problems-Fall-2026.ipynb) it is the tangent portfolio $\mathbf{w}_{\mathcal{T}}$.

Two things can go wrong, and the examples check both. If the target is so low that the solver returns almost nothing risky, dividing by a tiny sum magnifies roundoff, so the examples require $\theta$ above a small floor. If the target is so high that a cap $w_{i}\leq1$ binds, the solution has left the ray and the rescaled vector is not $\mathbf{w}_{\mathcal{T}}$, so the examples require every weight below its cap. Between those two limits, any target gives the same answer, which the L7a example confirms by sweeping the target and comparing the rescaled solutions.

What the shortcut does not give is the tangent point itself, the target at which $\theta=1$ and the investor neither lends nor borrows. The continuation below finds that target by repeating the solve, and it returns the same mix along the way.

___


## Algorithm and convergence
__Initialize__: Given the SIM mean vector $\hat{\mathbf{g}}_{\mathrm{SIM}}$, positive-definite covariance matrix $\hat{\mathbf{\Sigma}}_{g,\mathrm{SIM}}$, and risk-free rate $g_f$, require $\max_i\hat g_{\mathrm{SIM},i}>g_f$. Choose a growth-target increment $\Delta g>0$, a risky-fraction tolerance $0<\epsilon<1$, and a nonnegative integer iteration limit $\texttt{maxiter}$.

Set $g_{\star}^{(0)}\gets g_f$, $\mathbf{w}^{(0)}\gets\mathbf{0}$, $\theta^{(0)}\gets0$, and the iteration counter $k\gets0$. Set $g_L\gets g_f$, $g_U\gets\max_i\hat g_{\mathrm{SIM},i}$, $\texttt{bracketed}\gets\texttt{false}$, and $\texttt{converged}\gets\texttt{false}$.

While not $\texttt{converged}$ __do__:

1. Calculate the risky-fraction residual $r^{(k)}\gets\theta^{(k)}-1$.
2. Check for convergence:
   - If $|r^{(k)}|<\epsilon$ and no weight $w_{i}^{(k)}$ sits at its cap, __then__: set $\texttt{converged}\gets\texttt{true}$ and return the tangent-weight approximation $\mathbf{w}_{\mathcal{T}}\gets\mathbf{w}^{(k)}/\theta^{(k)}$, the target $g_{\star}^{(k)}$, and this status.
   - Otherwise, if $k\ge\texttt{maxiter}$, __then__: print a __warning__ that the risky-fraction tolerance was not met and return the current allocation with $\texttt{converged}=\texttt{false}$.
3. Update the target interval:
   - If $r^{(k)}<0$ and no weight $w_{i}^{(k)}$ sits at its cap, __then__: set $g_L\gets g_{\star}^{(k)}$; the current portfolio still lends.
   - Otherwise, __then__: set $g_U\gets g_{\star}^{(k)}$ and $\texttt{bracketed}\gets\texttt{true}$; we have crossed into borrowing or hit a cap.
4. Choose the next growth target:
   - If $\texttt{bracketed}$, __then__: set $g_{\star}^{(k+1)}\gets(g_L+g_U)/2$ to bisect the interval.
   - Otherwise, __then__: set $g_{\star}^{(k+1)}\gets\min(g_{\star}^{(k)}+\Delta g,\,g_U)$ to continue the upward sweep.
5. Solve [SIM-2](CHEME-5660-L7b-Reference-SIM-Portfolio-Problems-Fall-2026.ipynb) at $g_{\star}^{(k+1)}$, using $\mathbf{w}^{(k)}$ as the solver's starting point, and store the solution in $\mathbf{w}^{(k+1)}$. If the solver fails, __then__: print a __warning__ and stop with $\texttt{converged}=\texttt{false}$.
6. Calculate the risky fraction $\theta^{(k+1)}\gets\mathbf{1}^{\top}\mathbf{w}^{(k+1)}$ and risk-free fraction $w_f^{(k+1)}\gets1-\theta^{(k+1)}$.
7. Increment the solve counter: $k\gets k+1$.

The convergence check comes before the iteration limit, so an allocation that first meets tolerance on the last allowed solve is still a success. Either return stops the algorithm immediately, and a solver failure stops it with its own warning.

__Why does this work?__ Along the ray of [SIM-3](CHEME-5660-L7b-Reference-SIM-Portfolio-Problems-Fall-2026.ipynb), the risky fraction $\theta$ increases linearly with the target, so the residual $r=\theta-1$ changes sign exactly once, at the tangent point. The highest-growth asset alone meets the top target $g_{U}$ with $\theta\geq1$, so the upward sweep must cross $\theta=1$, and the crossing is then bracketed by $[g_{L},g_{U}]$. The sweep costs about $(g_{\mathcal{T}}-g_{f})/\Delta g$ solves. After that, bisection halves the bracket at every solve, and because $\theta$ is linear in the target on the ray, the bound on the residual halves too, so the bisection phase takes a number of solves that grows only logarithmically with the bracket width over the tolerance. The example reports the total. Each solve is a quadratic program warm-started from the previous solution.

__Why check the caps?__ A sweep step can overshoot to a target where a cap binds, and a capped allocation can have $\theta\approx1$ without being the tangent fund, for example the highest-growth asset alone at $g_{U}$. A capped allocation always has $\theta\geq1$, so steps 2 and 3 treat any allocation with a weight at its cap as the upper end of the bracket, and bisection moves the target back down to the ray. Applying the cap test in step 3 as well as step 2 matters in floating point, where a solver can return a capped weight a hair below one with $\theta$ a hair below one.

__What does the continuation add?__ Every uncapped iterate with $\theta^{(k)}>0$, rescaled to sum to one, is already $\mathbf{w}_{\mathcal{T}}$ up to solver tolerance, which is the single-solve shortcut. The continuation also returns the tangent target $g_{\star}\approx\hat{\mathbf{g}}_{\mathrm{SIM}}^{\top}\mathbf{w}_{\mathcal{T}}$, the point where the capital allocation line touches the frontier. The example checks its result against the single solve.

___


## How does this differ from the version shown in lecture?
The pseudocode above is the one from the L7a lecture with one added rule: steps 2 and 3 treat an allocation with a weight at its cap as the upper end of the bracket, and step 2 never accepts one. The lecture version stopped on the risky-fraction tolerance alone, and that test can be fooled. Take two assets with excess growth rates of $(1,-0.1)$ per year, unit variances, and a correlation of $-0.9$. A large increment $\Delta g$ carries the first solve straight to the top target, where the solver returns the highest-growth asset alone. Its risky fraction is exactly one, so the lecture version returns it as the tangent fund, although its Sharpe ratio is less than half that of the true fund. The cap condition sends that allocation to the borrowing branch, and bisection walks back down to the ray. On any path where no sweep step overshoots onto a cap, the two versions take the same solves.

__Can we converge faster?__ Yes, by using [SIM-3](CHEME-5660-L7b-Reference-SIM-Portfolio-Problems-Fall-2026.ipynb) in place of bisection. Along the ray, the risky fraction is linear in the target, so the first uncapped solve at target $g_{\star}^{(1)}$ with risky fraction $\theta^{(1)}$ predicts the tangent target directly:

$$
g_{\mathcal{T}} = g_{f}+\frac{g_{\star}^{(1)}-g_{f}}{\theta^{(1)}}
$$

One more solve at $g_{\mathcal{T}}$ confirms a risky fraction of one with no cap binding, so the search takes two solves instead of a dozen. The exception is a fund that holds a single asset, whose tangent allocation sits exactly at that asset's cap, so the confirming solve must instead accept a risky fraction just below one. Bisection never assumes the ray, which is why it is the safe default. The linear jump is exact only while no cap binds, the same condition the stopping test checks. The single solve of the examples is the limit of this idea, with no confirming solve at all.

___


## Summary
We found the long-only tangent portfolio from the risky and risk-free problem without searching for it directly.

> __Key Takeaways:__
>
> - **One risky direction:** We showed that scaling the weights scales the excess growth and the variance in step, so every solution of [SIM-2](CHEME-5660-L7b-Reference-SIM-Portfolio-Problems-Fall-2026.ipynb) below the tangent point is the same risky fund scaled by the risky fraction. One solve, rescaled to sum to one, gives the tangent portfolio, and this fund has the largest Sharpe ratio of any long-only portfolio.
> - **Continuation to the tangent point:** We stepped the growth target upward until lending turned into borrowing and bisected the crossing. Because the risky fraction grows linearly with the target along the ray, the crossing is unique, and each bisection halves the bracket and with it the bound on the residual.
> - **Convergence and stopping:** Our pseudocode returns immediately when the risky fraction meets tolerance, reports the iteration limit separately, and stops with a warning if a solve fails. A capped allocation can sit at a risky fraction of one without being the tangent fund, so the convergence check also requires that no cap binds, and a capped allocation always becomes the upper end of the bracket.

The allocator example runs both routes and checks that they agree.

___
